# Project 1: Dataset selection

## Video Game Sales with Ratings

The selected dataset contains **16,719 game/platform records** and 16 original columns. A game released on several platforms can appear in several rows. After excluding the game title and the global sales aggregate, there are **14 usable features**, including four continuous numerical variables and five categorical variables.

The proposed unsupervised learning task is to explore groups of game/platform releases with similar regional sales and review profiles, then examine their genres and platforms. No column is designated as a prediction target.

## Source

- Dataset: [Video Game Sales with Ratings, published by the Kaggle account rush4ratio](https://www.kaggle.com/datasets/rush4ratio/video-game-sales-with-ratings).
- Original data sources described by the publisher: VGChartz sales data and Metacritic ratings.
- Selected file: `Video_Games_Sales_as_at_22_Dec_2016.csv`, dataset version 2.
- Accessed on 6 October 2026.

This is a historical dataset with a file dated 22 December 2016. The sales figures describe this snapshot and should not be interpreted as current sales. Four records have release years later than 2016 and need inspection. The dataset is public, so the source link and download cell provide access to the original file.

## Usable features

Sales are reported in millions of units. These four regional sales variables are treated as continuous numerical measurements, despite their rounded representation. Review counts and release years are discrete numerical variables. Categorical variables remain categorical even if an encoder later represents them as numbers.

| Feature | Type | Description |
| --- | --- | --- |
| `Platform` | Categorical | Platform for this release, such as Wii, PS3, or X360 |
| `Year_of_Release` | Numerical, discrete | Release year for this platform |
| `Genre` | Categorical | Game genre |
| `Publisher` | Categorical | Publisher name |
| `NA_Sales` | Numerical, continuous | Sales in North America, in millions of units |
| `EU_Sales` | Numerical, continuous | Sales in Europe, in millions of units |
| `JP_Sales` | Numerical, continuous | Sales in Japan, in millions of units |
| `Other_Sales` | Numerical, continuous | Sales in other regions, in millions of units |
| `Critic_Score` | Numerical, score | Aggregated critic score on a 0 to 100 scale |
| `Critic_Count` | Numerical, discrete | Number of critics contributing to the critic score |
| `User_Score` | Numerical, score after parsing | Aggregated user score on a 0 to 10 scale |
| `User_Count` | Numerical, discrete | Number of users contributing to the user score |
| `Developer` | Categorical | Developer name |
| `Rating` | Categorical | ESRB age-rating label |

`Name` is retained for identifying and inspecting records, but excluded from the feature count. `Global_Sales` aggregates the four regional sales columns, subject to rounding, so it is excluded to avoid counting overall sales twice.

## Reproducible verification

The cells below download the selected version and inspect the original CSV. They preserve the raw data and create a separate copy solely to parse unavailable user scores. Outputs are saved in this notebook.

To rerun, use a Python 3 Jupyter kernel with pandas and an internet connection. A standard installation is `python -m pip install pandas jupyter`. Run the cells from top to bottom. No Kaggle account or API key was needed when verifying this public download.

In [1]:
import hashlib
import io
import platform
from urllib.error import URLError
from urllib.request import urlopen
from zipfile import BadZipFile, ZipFile

import pandas as pd
from IPython.display import display

# Pin the version and file hash so reruns inspect the same historical data.
download_url = (
    "https://www.kaggle.com/api/v1/datasets/download/"
    "rush4ratio/video-game-sales-with-ratings?datasetVersionNumber=2"
)
csv_name = "Video_Games_Sales_as_at_22_Dec_2016.csv"
expected_sha256 = "4ad802a9e51792e7a96b84da4501e5f64c97484d937ede9441d2e1145c1d7b03"

try:
    with urlopen(download_url, timeout=30) as response:
        archive_bytes = response.read()
    with ZipFile(io.BytesIO(archive_bytes)) as archive:
        csv_bytes = archive.read(csv_name)
except (URLError, TimeoutError, BadZipFile, KeyError) as error:
    raise RuntimeError(
        "Could not load the selected Kaggle file. Check the internet connection "
        "and the dataset download link, then rerun this cell."
    ) from error

snapshot_sha256 = hashlib.sha256(csv_bytes).hexdigest()
if snapshot_sha256 != expected_sha256:
    raise ValueError(
        "The downloaded CSV differs from the verified snapshot. "
        "Check the dataset version before using these results."
    )

raw = pd.read_csv(io.BytesIO(csv_bytes))
print(f"Rows: {len(raw):,}; original columns: {len(raw.columns)}")
print(f"CSV SHA-256: {snapshot_sha256}")
print(f"Python {platform.python_version()}; pandas {pd.__version__}")
display(raw[["Name", "Platform", "Year_of_Release", "Genre", "NA_Sales", "User_Score"]].head())

Rows: 16,719; original columns: 16
CSV SHA-256: 4ad802a9e51792e7a96b84da4501e5f64c97484d937ede9441d2e1145c1d7b03
Python 3.12.14; pandas 2.2.3


,Name,Platform,Year_of_Release,Genre,NA_Sales,User_Score
0,Wii Sports,Wii,2006.0,Sports,41.36,8
1,Super Mario Bros.,NES,1985.0,Platform,29.08,NaN
2,Mario Kart Wii,Wii,2008.0,Racing,15.68,8.3
3,Wii Sports Resort,Wii,2009.0,Sports,15.61,8
4,Pokemon Red/Pokemon Blue,GB,1996.0,Role-Playing,11.27,NaN


### Feature types and assignment criteria

Feature types below follow the meaning of each variable. For example, pandas reads the release year as a floating-point column because some years are missing, but a release year is still a discrete variable. The title and global sales aggregate are excluded before counting features.

In [2]:
feature_types = {
    "Platform": "Categorical",
    "Year_of_Release": "Numerical, discrete",
    "Genre": "Categorical",
    "Publisher": "Categorical",
    "NA_Sales": "Numerical, continuous",
    "EU_Sales": "Numerical, continuous",
    "JP_Sales": "Numerical, continuous",
    "Other_Sales": "Numerical, continuous",
    "Critic_Score": "Numerical, score",
    "Critic_Count": "Numerical, discrete",
    "User_Score": "Numerical, score",
    "User_Count": "Numerical, discrete",
    "Developer": "Categorical",
    "Rating": "Categorical",
}
usable_features = list(feature_types)
continuous_features = [name for name, kind in feature_types.items() if kind == "Numerical, continuous"]
categorical_features = [name for name, kind in feature_types.items() if kind == "Categorical"]

# Keep the declared feature set aligned with the exclusions used in this report.
assert set(usable_features) == set(raw.columns) - {"Name", "Global_Sales"}
duplicate_feature_pairs = [
    (left, right)
    for index, left in enumerate(usable_features)
    for right in usable_features[index + 1:]
    if raw[left].equals(raw[right])
]
assert not duplicate_feature_pairs, f"Exact duplicate features: {duplicate_feature_pairs}"

criteria = pd.DataFrame(
    {
        "Criterion": ["Rows", "Usable original features", "Continuous numerical features", "Categorical features"],
        "Required minimum": [2000, 10, 3, 2],
        "Observed": [len(raw), len(usable_features), len(continuous_features), len(categorical_features)],
    }
)
criteria["Meets requirement"] = criteria["Observed"] >= criteria["Required minimum"]
display(criteria.set_index("Criterion"))

feature_summary = pd.DataFrame(
    {
        "Feature": usable_features,
        "Type": [feature_types[name] for name in usable_features],
        "Raw pandas dtype": [str(raw[name].dtype) for name in usable_features],
    }
)
display(feature_summary.set_index("Feature"))

,Required minimum,Observed,Meets requirement
Criterion,,,
Rows,2000,16719,True
Usable original features,10,14,True
Continuous numerical features,3,4,True
Categorical features,2,5,True


,Type,Raw pandas dtype
Feature,,
Platform,Categorical,object
Year_of_Release,"Numerical, discrete",float64
Genre,Categorical,object
Publisher,Categorical,object
NA_Sales,"Numerical, continuous",float64
EU_Sales,"Numerical, continuous",float64
JP_Sales,"Numerical, continuous",float64
Other_Sales,"Numerical, continuous",float64
Critic_Score,"Numerical, score",float64


### Missing and non-numerical values

The CSV contains 6,704 missing user scores and another 2,425 scores recorded as `tbd`. Those entries mean that a numerical score is unavailable, not that the score is zero. Together they account for 9,129 unavailable scores, or 54.6% of the rows. The conversion below changes only the separate inspection copy and rejects unexpected non-numerical values.

In [3]:
inspected = raw.copy()
# 'tbd' represents an unavailable rating and must not become a zero score.
inspected["User_Score"] = pd.to_numeric(
    inspected["User_Score"].replace("tbd", pd.NA), errors="raise"
)

missing_summary = pd.DataFrame(
    {
        "Raw missing": raw.isna().sum(),
        "Missing after score parsing": inspected.isna().sum(),
        "Missing after parsing (%)": (inspected.isna().mean() * 100).round(2),
    }
)
display(missing_summary)
print(f"User scores recorded as 'tbd': {raw['User_Score'].eq('tbd').sum():,}")
print(f"Rows complete across all 14 usable features: {len(inspected[usable_features].dropna()):,}")

,Raw missing,Missing after score parsing,Missing after parsing (%)
Name,2,2,0.01
Platform,0,0,0.00
Year_of_Release,269,269,1.61
Genre,2,2,0.01
Publisher,54,54,0.32
NA_Sales,0,0,0.00
EU_Sales,0,0,0.00
JP_Sales,0,0,0.00
Other_Sales,0,0,0.00
Global_Sales,0,0,0.00


User scores recorded as 'tbd': 2,425
Rows complete across all 14 usable features: 6,825


### Duplicate records and release years

There are no exact duplicate rows. Eight records belong to repeated, non-missing game-title/platform pairs and need inspection before deciding whether they refer to the same release. Repeated titles on different platforms are expected and should remain separate observations. Four release-year entries exceed the year of the file's snapshot.

In [4]:
print(f"Exact duplicate rows: {raw.duplicated().sum():,}")
named_records = raw.loc[raw["Name"].notna()]
repeated_pairs = named_records.loc[named_records.duplicated(["Name", "Platform"], keep=False)]
print(f"Records in repeated game-title/platform pairs: {len(repeated_pairs):,}")
display(repeated_pairs[["Name", "Platform", "Year_of_Release", "Publisher", "Developer"]])

future_years = raw.loc[raw["Year_of_Release"] > 2016]
print(f"Release-year entries later than 2016: {len(future_years):,}")
display(future_years[["Name", "Platform", "Year_of_Release"]])

print("Distinct non-missing categorical values:")
display(raw[categorical_features].nunique().rename("Distinct values").to_frame())

Exact duplicate rows: 0
Records in repeated game-title/platform pairs: 8


,Name,Platform,Year_of_Release,Publisher,Developer
604,Madden NFL 13,PS3,2012.0,Electronic Arts,EA Tiburon
1190,Need for Speed: Most Wanted,X360,2012.0,Electronic Arts,EA Canada
1591,Need for Speed: Most Wanted,X360,2005.0,Electronic Arts,EA Canada
1745,Sonic the Hedgehog,PS3,2006.0,Sega,Sonic Team
4127,Sonic the Hedgehog,PS3,NaN,NaN,Sonic Team
5973,Need for Speed: Most Wanted,PC,2005.0,Electronic Arts,Black Box
11716,Need for Speed: Most Wanted,PC,2012.0,Electronic Arts,Black Box
16233,Madden NFL 13,PS3,2012.0,Electronic Arts,EA Tiburon


Release-year entries later than 2016: 4


,Name,Platform,Year_of_Release
5936,Imagine: Makeup Artist,DS,2020.0
14086,Phantasy Star Online 2 Episode 4: Deluxe Package,PS4,2017.0
16222,Phantasy Star Online 2 Episode 4: Deluxe Package,PSV,2017.0
16385,Brothers Conflict: Precious Baby,PSV,2017.0


Distinct non-missing categorical values:


,Distinct values
Platform,31
Genre,12
Publisher,581
Developer,1696
Rating,8


## Expected preprocessing

1. **Parse numerical fields.** Convert `tbd` user scores to missing values and parse valid scores as numbers. Validate score ranges, non-negative sales and counts, and integer-valued years and counts. Inspect release years later than the snapshot date before making corrections.
2. **Handle missing values explicitly.** Examine missingness by platform, genre, and release year. A possible starting approach is median imputation with missingness indicators for numerical fields and an explicit missing category for categorical fields. Imputed values must remain distinguishable from observed values. Compare with complete-case analysis, which retains 6,825 rows, since unavailable reviews may not be random.
3. **Inspect duplicate releases.** Review the eight records with repeated title/platform pairs. Remove records only when they represent confirmed duplicate observations. Do not merge releases solely because they share a game title.
4. **Encode categorical features.** Check labels before one-hot encoding. Publisher and developer have 581 and 1,696 distinct non-missing values, respectively. Inspect rare labels and use a documented grouping rule if needed. Omitting these two fields would still leave 12 original features.
5. **Transform and scale numerical features.** Inspect the strongly skewed regional sales and review counts. Consider `log1p` for non-negative sales and counts, then scale numerical features so large sales figures or counts do not dominate distances. Do not replace zero sales with missing values.
6. **Exclude identifiers and redundant data.** Keep `Name` for interpreting records and exclude it from modeling. Exclude `Global_Sales` because it repeats the regional sales aggregate. Review correlations and the relative influence of encoded categorical and numerical variables before choosing a clustering method.
7. **Keep evaluation separate from preprocessing.** If later evaluation uses held-out rows, fit imputation, encoding, scaling, and any dimensionality reduction only on the training rows.

These are proposed operations for the later analysis. The verification cells preserve the original data and parse user scores only in a separate inspection copy.

## Suitability and limitations

The dataset meets the measurable size, feature-count, and type-diversity criteria. It requires active preprocessing because of missing fields, unavailable user scores stored as strings, and categorical labels. It is real game-market data rather than a standard toy dataset.

The snapshot is historical, review coverage is incomplete, and the CSV should not be assumed to represent every game or every sale. A cluster would describe patterns in these recorded game/platform releases, not establish that a genre or a review score causes higher sales.